# Exploração · `bronze.atendimentos`

**Status: exploração concluída.** Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade de `atendimentos` e a regra de limpeza de cada um, antes de escrever a Silver.

A tabela vem do `atendimentos.json` (~319 mil chamados do call center). Cada chamado tem campos simples (`protocolo`, `id_uc`, `canal`, `assunto`, `subassunto`, `data_abertura`, `data_fechamento`, `nota_satisfacao`) e um campo aninhado, o `historico`, que é uma **lista de tratativas** (cada uma com `sequencia`, `data`, `acao` e `canal`). Na Bronze, o JSON foi lido com `primitivesAsString`: os valores simples são texto, e a estrutura (lista de registros) foi preservada.

**Como usar:** rode de cima para baixo (Run all). Cada célula só depende das anteriores. As conversões usam `try_cast` e `try_to_timestamp`, que devolvem nulo em vez de dar erro (o `cast` comum falha no serverless).

**O que já sabemos de `leituras`, `faturas` e `medidores` e vamos testar aqui:**
- cabeçalho repetido como linha de dado (achado em `faturas` e `unidades_consumidoras`);
- órfãs: ids fora do padrão e UCs que não existem no cadastro (`UC` com 7 dígitos);
- datas em até 4 formatos, com o hífen sendo **mês primeiro**, e a sentinela `9999-12-31`;
- códigos de ausência (`N/A`, `NULL`, `-1`) no lugar de nulo;
- duas "formas" de a mesma informação aparecer (nulo real e texto).

Resultado, em uma linha: **nada disso apareceu em `atendimentos`**. A tabela é consistente, e a anomalia está no volume de julho de 2025 (seção 10.1).

| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Quais colunas e qual estrutura do `historico`? |
| 2 | Nulos e texto no lugar de dado | Onde há ausência, e de que forma? |
| 3 | Categorias | `canal`, `assunto`, `subassunto` e `acao` têm grafias diferentes? |
| 4 | Formatos de data | As datas seguem um único padrão? Há sentinela? |
| 5 | Duplicatas e ids | O protocolo se repete? Há cabeçalho repetido? |
| 6 | Integridade com as UCs | Toda UC do chamado existe? |
| 7 | Datas: ordem dia/mês e coerência | Qual formato é qual? A abertura vem antes do fechamento? |
| 8 | Nota de satisfação | Formato, faixa e relação com o chamado aberto |
| 9 | Histórico (tratativas) | Sequência, datas e última ação fazem sentido? |
| 10 | Distribuição no tempo e por UC | O volume é regular? Há algum mês fora do padrão? |
| 10.1 a 10.3 | O pico de julho de 2025 | O que sobe, em que tabelas, e por quê? |
| 11 | Resumo | Quais regras propor para a Silver? |

**Sobre os números:** as seções 1 a 5 rodam sobre a tabela bruta. A partir da Seção 5, `at_d` é a tabela sem cópias exatas por protocolo. Da Seção 6, `limpo` é `at_d` restrita às UCs que existem de verdade (as que têm leitura). Os números em "Achado" vêm da execução sobre os dados atuais; confira ao rodar de novo.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

atend = spark.table(f"{bronze_schema}.atendimentos")

## 1. Visão geral

**O que explora:** colunas, tipos, estrutura do `historico` e uma amostra. Esperado, pela amostra do arquivo: `protocolo`, `id_uc`, `canal`, `assunto`, `subassunto`, `data_abertura`, `data_fechamento`, `nota_satisfacao`, `historico` (lista de registros com `sequencia`, `data`, `acao`, `canal`) e as duas colunas de auditoria (`source_file`, `ingestion_timestamp`).

**Achado:** 318.871 chamados. O schema real bate com o esperado: `protocolo`, `id_uc`, `canal`, `assunto`, `subassunto`, `data_abertura`, `data_fechamento`, `nota_satisfacao` (todos texto) e `historico`, uma lista de registros com `acao`, `canal`, `data` e `sequencia` (também texto). Não há colunas inesperadas.

In [0]:
atend.printSchema()
atend.limit(5).display()
print(f"{atend.count():,} linhas")

root
 |-- assunto: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- data_abertura: string (nullable = true)
 |-- data_fechamento: string (nullable = true)
 |-- historico: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- acao: string (nullable = true)
 |    |    |-- canal: string (nullable = true)
 |    |    |-- data: string (nullable = true)
 |    |    |-- sequencia: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- nota_satisfacao: string (nullable = true)
 |-- protocolo: string (nullable = true)
 |-- subassunto: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



assunto,canal,data_abertura,data_fechamento,historico,id_uc,nota_satisfacao,protocolo,subassunto,source_file,ingestion_timestamp
RELIGACAO,SITE,2022-07-02 15:51:00,2022-07-10 14:51:00,"List(List(RETORNO AO CLIENTE, SITE, 2022-07-10T14:51:00, 1))",UC142511,3.9,AT000000000,URGENCIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
FALTA DE ENERGIA,SITE,2022-07-07 23:25:00,2022-07-17 10:25:00,"List(List(RETORNO AO CLIENTE, SITE, 2022-07-12T16:25:00, 1), List(DESPACHO, SITE, 2022-07-17T10:25:00, 2))",UC100661,4.1,AT000000001,INTERRUPCAO TOTAL,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
LIGACAO NOVA,WHATSAPP,2022-07-11 10:48:00,2022-07-13 21:48:00,"List(List(RETORNO AO CLIENTE, WHATSAPP, 2022-07-12T05:48:00, 1), List(ABERTURA, WHATSAPP, 2022-07-13T01:48:00, 2), List(ENCERRAMENTO, WHATSAPP, 2022-07-13T21:48:00, 3))",UC108189,4.4,AT000000002,SOLICITACAO,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
DANO ELETRICO,TELEFONE,2022-07-09 17:11:00,2022-07-16 05:11:00,"List(List(ANALISE, TELEFONE, 2022-07-11T21:11:00, 1), List(ABERTURA, TELEFONE, 2022-07-14T01:11:00, 2), List(RETORNO AO CLIENTE, TELEFONE, 2022-07-16T05:11:00, 3))",UC104975,4.0,AT000000003,VISTORIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z
FATURA,WHATSAPP,2022-07-23 15:44:00,2022-07-27 19:44:00,"List(List(ABERTURA, WHATSAPP, 2022-07-25T17:44:00, 1), List(ENCERRAMENTO, WHATSAPP, 2022-07-27T19:44:00, 2))",UC108340,4.0,AT000000004,SEGUNDA VIA,dbfs:/Volumes/v_energy/bronze/landing/atendimentos.json,2026-10-24T19:28:18.799Z


318,871 linhas


## 2. Nulos e texto no lugar de dado

**O que explora:** quantas células vazias há por coluna e quais colunas guardam texto de ausência (`N/A`, `NULL`, `-1`, vazio, `9999-12-31`). Também mede quantos chamados têm o `historico` nulo ou vazio.

**Como ler:** `data_fechamento` nula pode ser um chamado ainda aberto (nulo de negócio). `nota_satisfacao` nula pode ser chamado sem avaliação (também nulo de negócio). Já um texto como `N/A` no lugar do nulo é uma segunda forma de ausência, como em `leitura_anterior`.

**Achado:**
- Só `nota_satisfacao` tem nulos: **9.566** (3,0% dos chamados). As demais colunas têm 0.
- A célula de texto de ausência não imprimiu nada: nenhuma coluna usa `N/A`, `NULL`, `-1` ou `9999-12-31` escritos como texto.
- O `historico` nunca é nulo nem vazio. Cada chamado tem 1, 2 ou 3 tratativas, em partes quase iguais (106.717, 106.322 e 105.832 chamados).

In [0]:
colunas_simples = [c for c, t in atend.dtypes if not t.startswith("array") and not t.startswith("struct")]

# Nulos reais por coluna
atend.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_simples]).display()

# Texto de ausência no lugar do nulo
codigos = ["N/A", "NULL", "NONE", "NAN", "", "-1", "9999-12-31"]
for c in colunas_simples:
    n = atend.filter(F.upper(F.trim(F.col(c).cast("string"))).isin(*codigos)).count()
    if n:
        print(f"{c}: {n:,} valores de ausência escritos como texto")

assunto,canal,data_abertura,data_fechamento,id_uc,nota_satisfacao,protocolo,subassunto,source_file,ingestion_timestamp
0,0,0,0,0,9566,0,0,0,0


In [0]:
# Tamanho do historico: nulo, vazio e quantas tratativas por chamado
tam = F.when(F.col("historico").isNull(), 0).otherwise(F.size("historico"))
print("historico nulo:", atend.filter(F.col("historico").isNull()).count(),
      "| vazio:", atend.filter(F.size("historico") == 0).count())
atend.groupBy(tam.alias("qtd_tratativas")).count().orderBy("qtd_tratativas").display()

historico nulo: 0 | vazio: 0


qtd_tratativas,count
1,106717
2,106322
3,105832


## 3. Categorias

**O que explora:** valores distintos de `canal`, `assunto`, `subassunto` e das tratativas (`acao`, `canal` do histórico), procurando o mesmo valor escrito de formas diferentes (caixa, acento, espaço). A combinação `assunto` × `subassunto` mostra se os pares fazem sentido.

**Como ler:** se o mesmo assunto aparece como `FALTA DE ENERGIA` e `Falta de Energia`, é preciso padronizar. Se um subassunto aparece sob vários assuntos que não combinam, vale registrar.

**Achado:** sem variações de grafia.
- 5 canais (TELEFONE 64.037, APP 64.002, WHATSAPP 63.758, SITE 63.670, PRESENCIAL 63.404).
- 6 assuntos. `FALTA DE ENERGIA` concentra 31,8% dos chamados (101.306); cada um dos outros cinco tem ~13,6% (de 43.340 a 43.624).
- 17 pares assunto × subassunto, todos coerentes (cada subassunto aparece sob um único assunto, exceto `VISTORIA`, que aparece em três: `DANO ELETRICO`, `GERACAO DISTRIBUIDA` e `LIGACAO NOVA`).
- 5 ações no histórico (ABERTURA 127.796, RETORNO AO CLIENTE 127.484, DESPACHO 127.417, ENCERRAMENTO 127.256, ANALISE 126.904) e os mesmos 5 canais.

In [0]:
for c in ["canal", "assunto", "subassunto"]:
    print(c)
    atend.groupBy(c).count().orderBy(F.desc("count")).show(50, truncate=False)

canal
+----------+-----+
|canal     |count|
+----------+-----+
|TELEFONE  |64037|
|APP       |64002|
|WHATSAPP  |63758|
|SITE      |63670|
|PRESENCIAL|63404|
+----------+-----+

assunto
+-------------------+------+
|assunto            |count |
+-------------------+------+
|FALTA DE ENERGIA   |101306|
|FATURA             |43624 |
|DANO ELETRICO      |43582 |
|LIGACAO NOVA       |43520 |
|GERACAO DISTRIBUIDA|43499 |
|RELIGACAO          |43340 |
+-------------------+------+

subassunto
+-------------------+-----+
|subassunto         |count|
+-------------------+-----+
|VISTORIA           |51106|
|INTERRUPCAO PARCIAL|34014|
|INTERRUPCAO TOTAL  |33652|
|OSCILACAO          |33640|
|URGENCIA           |21789|
|RESSARCIMENTO      |21695|
|APOS PAGAMENTO     |21551|
|SOLICITACAO        |14563|
|HOMOLOGACAO        |14454|
|PRAZO              |14412|
|CREDITOS           |14371|
|CONTESTACAO        |11022|
|SEGUNDA VIA        |10959|
|VALOR ALTO         |10856|
|PARCELAMENTO       |10787|
+-------

In [0]:
# Pares assunto x subassunto
atend.groupBy("assunto", "subassunto").count().orderBy("assunto", "subassunto").display()

assunto,subassunto,count
DANO ELETRICO,RESSARCIMENTO,21695
DANO ELETRICO,VISTORIA,21887
FALTA DE ENERGIA,INTERRUPCAO PARCIAL,34014
FALTA DE ENERGIA,INTERRUPCAO TOTAL,33652
FALTA DE ENERGIA,OSCILACAO,33640
FATURA,CONTESTACAO,11022
FATURA,PARCELAMENTO,10787
FATURA,SEGUNDA VIA,10959
FATURA,VALOR ALTO,10856
GERACAO DISTRIBUIDA,CREDITOS,14371


In [0]:
# Categorias dentro do histórico
trat = atend.select(F.explode("historico").alias("t"))
print("acao")
trat.groupBy("t.acao").count().orderBy(F.desc("count")).show(50, truncate=False)
print("canal da tratativa")
trat.groupBy("t.canal").count().orderBy(F.desc("count")).show(50, truncate=False)

acao
+------------------+------+
|acao              |count |
+------------------+------+
|ABERTURA          |127796|
|RETORNO AO CLIENTE|127484|
|DESPACHO          |127417|
|ENCERRAMENTO      |127256|
|ANALISE           |126904|
+------------------+------+

canal da tratativa
+----------+------+
|canal     |count |
+----------+------+
|APP       |128407|
|TELEFONE  |127712|
|WHATSAPP  |127044|
|SITE      |126965|
|PRESENCIAL|126729|
+----------+------+



## 4. Formatos de data

**O que explora:** troca cada dígito por `9` para mostrar quais formatos existem em `data_abertura`, `data_fechamento` e na `data` de cada tratativa. Mais de um padrão na mesma coluna significa datas misturadas. Também conta valores que começam com `9999` (data sentinela).

**Como ler:** na amostra do arquivo, `data_abertura` e `data_fechamento` aparecem como `2022-07-02 15:51:00` (data e hora separadas por espaço) e a `data` da tratativa como `2022-07-10T14:51:00` (com `T`). A ordem dia/mês dos formatos que não forem ISO é conferida na Seção 7.

**Achado:** **formato único** nas três colunas, sem sentinela.
- `data_abertura` e `data_fechamento`: `yyyy-MM-dd HH:mm:ss` em todas as 318.871 linhas. Nenhum valor começa com `9999`.
- `data` da tratativa: `yyyy-MM-dd'T'HH:mm:ss` em todas as 636.857 tratativas.

In [0]:
def padrao(c):
    return F.regexp_replace(F.col(c).cast("string"), "[0-9]", "9")

for c in ["data_abertura", "data_fechamento"]:
    print(c)
    (atend.withColumn("padrao", padrao(c))
          .groupBy("padrao").count().orderBy(F.desc("count")).show(truncate=False))
    print(f"{c} começando com 9999 (sentinela):", atend.filter(F.col(c).startswith("9999")).count())

data_abertura
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99 99:99:99|318871|
+-------------------+------+

data_abertura começando com 9999 (sentinela): 0
data_fechamento
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99 99:99:99|318871|
+-------------------+------+

data_fechamento começando com 9999 (sentinela): 0


In [0]:
print("data da tratativa (historico.data)")
(atend.select(F.explode_outer("historico").alias("t"))
      .withColumn("padrao", F.regexp_replace(F.col("t.data").cast("string"), "[0-9]", "9"))
      .groupBy("padrao").count().orderBy(F.desc("count")).show(truncate=False))

data da tratativa (historico.data)
+-------------------+------+
|padrao             |count |
+-------------------+------+
|9999-99-99T99:99:99|636857|
+-------------------+------+



## 5. Duplicatas e ids

**O que explora:**
- se `protocolo` se repete, e se as repetições são cópias exatas (linhas idênticas fora das colunas de auditoria) ou chamados diferentes com o mesmo protocolo;
- se o protocolo segue o padrão `AT` + 9 dígitos, e se há protocolos nulos;
- se existe **cabeçalho repetido** como linha de dado (o valor é o nome da coluna). É um JSON e não deveria ter, mas `faturas` e `unidades_consumidoras` tiveram, e `atendimentos` ainda não foi testada;
- se há chamados repetidos com **protocolos diferentes** (mesma UC, mesma abertura e mesmo assunto), o que em `leituras` e `faturas` era sinal de linha espúria.

**Achado:** nada a tratar.
- 318.871 linhas, 318.871 protocolos distintos e 318.871 linhas distintas (sem as colunas de auditoria): sem cópias.
- Protocolo nulo: 0. Fora do padrão `AT` + 9 dígitos: 0. Cabeçalho repetido: 0.
- Protocolos com mais de uma linha: 0. Combinações (UC, abertura, assunto) com mais de um protocolo: 0.

In [0]:
total = atend.count()
ids_unicos = atend.select("protocolo").distinct().count()
cols = [c for c in atend.columns if c not in ("source_file", "ingestion_timestamp")]
linhas_unicas = atend.select(cols).distinct().count()

print(f"linhas: {total:,}")
print(f"protocolos distintos: {ids_unicos:,}  -> repetidos: {total - ids_unicos:,}")
print(f"linhas distintas (sem auditoria): {linhas_unicas:,}  (igual aos ids = cópias exatas)")
print("protocolo nulo:", atend.filter(F.col("protocolo").isNull()).count())
print("protocolo fora do padrão AT + 9 dígitos:", atend.filter(~F.coalesce(F.col("protocolo").rlike("^AT[0-9]{9}$"), F.lit(False))).count())
print("cabeçalho repetido (protocolo = 'protocolo'):", atend.filter(F.col("protocolo") == "protocolo").count())

linhas: 318,871
protocolos distintos: 318,871  -> repetidos: 0
linhas distintas (sem auditoria): 318,871  (igual aos ids = cópias exatas)
protocolo nulo: 0
protocolo fora do padrão AT + 9 dígitos: 0
cabeçalho repetido (protocolo = 'protocolo'): 0


In [0]:
# Protocolos repetidos: o conteúdo é igual?
dups = atend.groupBy("protocolo").count().filter("count > 1")
print(f"protocolos com mais de uma linha: {dups.count():,}")
(atend.join(dups.select("protocolo"), "protocolo")
      .orderBy("protocolo").limit(20).display())

protocolos com mais de uma linha: 0


protocolo,assunto,canal,data_abertura,data_fechamento,historico,id_uc,nota_satisfacao,subassunto,source_file,ingestion_timestamp


In [0]:
# Sem as cópias exatas por protocolo; daqui em diante usamos at_d
at_d = atend.dropDuplicates(["protocolo"])
print("linhas em at_d:", f"{at_d.count():,}")

# Chamados idênticos com protocolos diferentes (mesma UC, mesma abertura, mesmo assunto)
rep = at_d.groupBy("id_uc", "data_abertura", "assunto").count().filter("count > 1")
print("combinações (UC, abertura, assunto) com mais de um protocolo:", f"{rep.count():,}")
(at_d.join(rep.select("id_uc", "data_abertura", "assunto"), ["id_uc", "data_abertura", "assunto"])
     .orderBy("id_uc", "data_abertura").limit(20).display())

linhas em at_d: 318,871
combinações (UC, abertura, assunto) com mais de um protocolo: 0


id_uc,data_abertura,assunto,canal,data_fechamento,historico,nota_satisfacao,protocolo,subassunto,source_file,ingestion_timestamp


## 6. Integridade com as UCs

**O que explora:** se o `id_uc` do chamado existe. Classifica cada chamado em quatro dimensões: UC nula, formato `UC` + 6 dígitos, existência em `bronze.unidades_consumidoras` (sem a linha de cabeçalho repetido) e existência entre as UCs que **têm leitura**.

**Por que as duas referências:** o cadastro da Bronze tem 200 UCs de teste que a Aimê tirou da Silver dela e que **não** aparecem em leituras nem em faturas. As UCs com leitura (45.000) são a base real do case. Um chamado cuja UC está no cadastro mas não tem leitura aponta para uma UC de teste.

**Como ler:** a combinação mais frequente deve ser "UC com formato certo, no cadastro e com leitura". Qualquer outra é órfã ou suspeita. Em `faturas` as órfãs tinham `id_uc` de 7 dígitos.

Ao final, `limpo` mantém só os chamados cuja UC tem leitura.

**Achado:** todos os 318.871 chamados têm UC não nula, no padrão `UC` + 6 dígitos, existente no cadastro **e** com leitura. Não há órfãs nem chamados em UCs de teste. `limpo` = `at_d` = 318.871 linhas, e 44.940 das 45.000 UCs têm ao menos um chamado.

In [0]:
ucs_cadastro = (spark.table(f"{bronze_schema}.unidades_consumidoras")
                .filter(F.col("id_uc") != "id_uc")
                .select("id_uc").distinct().withColumn("no_cadastro", F.lit(True)))
ucs_com_leitura = (spark.table(f"{silver_schema}.leituras")
                   .select("id_uc").distinct().withColumn("tem_leitura", F.lit(True)))

classif = (at_d
    .withColumn("uc_nula", F.col("id_uc").isNull())
    .withColumn("fmt_uc_ok", F.col("id_uc").rlike("^UC[0-9]{6}$"))
    .join(ucs_cadastro, "id_uc", "left")
    .join(ucs_com_leitura, "id_uc", "left")
    .withColumn("no_cadastro", F.coalesce("no_cadastro", F.lit(False)))
    .withColumn("tem_leitura", F.coalesce("tem_leitura", F.lit(False))))

(classif.groupBy("uc_nula", "fmt_uc_ok", "no_cadastro", "tem_leitura")
        .count().orderBy(F.desc("count")).display())

uc_nula,fmt_uc_ok,no_cadastro,tem_leitura,count
false,true,true,true,318871


In [0]:
# Tabela restrita às UCs reais (as que têm leitura)
limpo = classif.filter("tem_leitura").select(*at_d.columns)
print("chamados em at_d:", f"{at_d.count():,}", "| em limpo:", f"{limpo.count():,}")
print("UCs distintas com chamado:", f"{limpo.select('id_uc').distinct().count():,}", "(de 45.000)")

chamados em at_d: 318,871 | em limpo: 318,871
UCs distintas com chamado: 44,940 (de 45.000)


## 7. Datas: ordem dia/mês e coerência

**O que explora:** para cada padrão de data que não é ISO, testa os dois formatos possíveis (dia primeiro e mês primeiro) contra um **gabarito independente**: o histórico do próprio chamado. Pela amostra, a última tratativa tem a mesma data e hora do `data_fechamento`, e a primeira tratativa vem depois da abertura. Se isso valer, as regras são:
- `data_abertura <= primeira tratativa`;
- `data_fechamento == última tratativa`.

**Como ler a 7.1:** para cada padrão e formato, `nao_converteu` deve ser 0 e a regra deve valer em quase 100% das linhas. O formato errado falha: converte menos linhas ou bate pouco com o gabarito. Em `leituras` e `faturas`, o hífen era mês primeiro e o ponto e a barra eram dia primeiro, mas aqui isso **não pode ser assumido**: o resultado diz.

**7.2:** converte as datas com o mapeamento da 7.1 (o `DATA_FMT` abaixo começa com a hipótese herdada das outras tabelas; ajuste se a 7.1 mostrar outra coisa) e mede a coerência: abertura depois do fechamento, datas fora do período da base (2022-07 a 2026-06) e conversões que falharam.

**Achado:** como só existe um formato, não havia ordem dia/mês a descobrir, mas o gabarito do histórico confirma a conversão em 100%:
- Nenhuma data de tratativa deixou de converter, e todos os 318.871 chamados têm gabarito.
- `data_abertura <= primeira tratativa`: 318.871 de 318.871. `data_fechamento == última tratativa`: 318.871 de 318.871.
- Nenhuma conversão falhou e nenhum fechamento vem antes da abertura. **Nenhum chamado está aberto**: todos têm `data_fechamento`.
- Aberturas de 2022-07-01 a 2026-06-30. Os fechamentos chegam a 2026-07-10 (chamados abertos no fim de junho, depois do fim da base).

In [0]:
# Gabarito: primeira e última tratativa de cada chamado, convertidas como timestamp
g = (at_d
     .withColumn("t", F.expr("transform(historico, x -> try_cast(x.data as timestamp))"))
     .withColumn("hist_min", F.array_min("t"))
     .withColumn("hist_max", F.array_max("t")))

print("datas de tratativa que não converteram:",
      g.select(F.sum(F.expr("size(filter(t, x -> x IS NULL))"))).first()[0])
print("chamados com gabarito (historico com data válida):", g.filter("hist_min IS NOT NULL").count())

DATA_CANDIDATOS = {
    "9999-99-99": ["yyyy-MM-dd"],
    "99-99-9999": ["dd-MM-yyyy", "MM-dd-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}
SUFIXOS = {"": "", " 99:99:99": " HH:mm:ss", "T99:99:99": "'T'HH:mm:ss", " 99:99": " HH:mm"}

def testar_formatos(coluna, regras):
    """Para cada padrão observado, testa os formatos candidatos contra as regras do gabarito."""
    d = (g.withColumn("_p", padrao(coluna))
          .filter("hist_min IS NOT NULL")
          .filter(~F.col(coluna).startswith("9999")))
    padroes = [r["_p"] for r in d.select("_p").distinct().collect() if r["_p"]]
    saida = None
    for p in sorted(padroes):
        suf = SUFIXOS.get(p[10:])
        if p[:10] not in DATA_CANDIDATOS or suf is None:
            print(f"padrão sem formato candidato: {p!r} (ajuste DATA_CANDIDATOS e SUFIXOS)")
            continue
        for fd in DATA_CANDIDATOS[p[:10]]:
            fmt = fd + suf
            conv = F.expr(f"try_to_timestamp({coluna}, '{fmt.replace(chr(39), chr(92) + chr(39))}')")
            aggs = [F.lit(p).alias("padrao"), F.lit(fmt).alias("formato"),
                    F.count("*").alias("linhas"),
                    F.sum(conv.isNull().cast("int")).alias("nao_converteu")]
            for i, (nome, regra) in enumerate(regras.items(), 1):
                aggs.append(F.sum(regra(conv).cast("int")).alias(f"regra_{i}"))
            r = d.filter(F.col("_p") == p).agg(*aggs)
            saida = r if saida is None else saida.unionByName(r)
    for i, nome in enumerate(regras, 1):
        print(f"regra_{i}: {nome}")
    return saida

datas de tratativa que não converteram: 0
chamados com gabarito (historico com data válida): 318871


### 7.1 Teste dos formatos
Em cada coluna, procure o formato com `nao_converteu = 0` e a regra próxima de `linhas`.

In [0]:
res = testar_formatos("data_abertura", {"abertura <= primeira tratativa": lambda c: c <= F.col("hist_min")})
if res is not None: res.display()

regra_1: abertura <= primeira tratativa


padrao,formato,linhas,nao_converteu,regra_1
9999-99-99 99:99:99,yyyy-MM-dd HH:mm:ss,318871,0,318871


In [0]:
res = testar_formatos("data_fechamento", {
    "fechamento == última tratativa": lambda c: c == F.col("hist_max"),
    "fechamento >= última tratativa": lambda c: c >= F.col("hist_max"),
})
if res is not None: res.display()

regra_1: fechamento == última tratativa
regra_2: fechamento >= última tratativa


padrao,formato,linhas,nao_converteu,regra_1,regra_2
9999-99-99 99:99:99,yyyy-MM-dd HH:mm:ss,318871,0,318871,318871


### 7.2 Conversão e coerência
Ajuste `DATA_FMT` se a 7.1 apontar outra ordem. A sentinela `9999-...` em `data_fechamento` vira nulo (chamado aberto).

In [0]:
# Hipótese inicial, herdada de leituras e faturas (hífen = mês primeiro). Ajuste se a 7.1 disser outra coisa.
DATA_FMT = {"9999-99-99": "yyyy-MM-dd", "99-99-9999": "MM-dd-yyyy", "99.99.9999": "dd.MM.yyyy", "99/99/9999": "dd/MM/yyyy"}

def converter(coluna):
    p = padrao(coluna)
    expr_ = None
    for dp, fd in DATA_FMT.items():
        for resto, fh in SUFIXOS.items():
            fmt = fd + fh
            val = F.expr(f"try_to_timestamp({coluna}, '{fmt.replace(chr(39), chr(92) + chr(39))}')")
            cond = p == (dp + resto)
            expr_ = F.when(cond, val) if expr_ is None else expr_.when(cond, val)
    return expr_

at_dt = (limpo
    .withColumn("abertura_ts", converter("data_abertura"))
    .withColumn("fechamento_ts",
                F.when(F.col("data_fechamento").startswith("9999"), F.lit(None).cast("timestamp"))
                 .otherwise(converter("data_fechamento")))
    .withColumn("aberto", F.col("fechamento_ts").isNull()))

print("data_abertura preenchida e não convertida:", at_dt.filter("data_abertura IS NOT NULL AND abertura_ts IS NULL").count(), "(esperado: 0)")
print("data_fechamento preenchida (fora a sentinela) e não convertida:",
      at_dt.filter("data_fechamento IS NOT NULL AND NOT startswith(data_fechamento, '9999') AND fechamento_ts IS NULL").count(), "(esperado: 0)")
print("fechamento antes da abertura:", at_dt.filter("fechamento_ts < abertura_ts").count(), "(esperado: 0)")
at_dt.agg(F.min("abertura_ts"), F.max("abertura_ts"), F.min("fechamento_ts"), F.max("fechamento_ts")).display()
print("chamados abertos (sem fechamento):", at_dt.filter("aberto").count(), "de", f"{at_dt.count():,}")

gab = g.select("protocolo", "hist_min", "hist_max")
chk = at_dt.join(gab, "protocolo").filter("hist_min IS NOT NULL")
print("abertura <= primeira tratativa:", chk.filter("abertura_ts <= hist_min").count(), "de", chk.count())
print("fechamento == última tratativa:", chk.filter("fechamento_ts = hist_max").count(), "de", chk.filter("fechamento_ts IS NOT NULL").count())

data_abertura preenchida e não convertida: 0 (esperado: 0)
data_fechamento preenchida (fora a sentinela) e não convertida: 0 (esperado: 0)
fechamento antes da abertura: 0 (esperado: 0)


min(abertura_ts),max(abertura_ts),min(fechamento_ts),max(fechamento_ts)
2022-07-01T00:00:00.000Z,2026-06-30T23:56:00.000Z,2022-07-01T08:59:00.000Z,2026-07-10T14:27:00.000Z


chamados abertos (sem fechamento): 0 de 318,871
abertura <= primeira tratativa: 318871 de 318871
fechamento == última tratativa: 318871 de 318871


## 8. Nota de satisfação

**O que explora:** o formato (ponto ou vírgula como separador decimal), valores que não viram número, a faixa (mínimo e máximo) e a distribuição. Também a relação com o chamado: uma nota só deveria existir em chamado **fechado**, e a proporção de fechados avaliados diz quanto da nota é nulo de negócio.

**Como ler:** o contrato da Gold deixa "escala a confirmar". A amostra tem `3.9`, então a nota pode ter decimais. Se a faixa for de 1 a 5, valores fora dela são erros. Nota em chamado aberto é uma incoerência a registrar.

**Achado:**
- 309.305 notas preenchidas e 9.566 nulas (3,0%). Todas as nulas são de chamados fechados (não há chamado aberto), então o nulo significa "não avaliado" (nulo de negócio).
- Nenhuma nota é texto, e nenhuma usa vírgula decimal.
- Faixa de 1,3 a 5,0, média 4,043, com casas decimais (distribuição em torno de 4,1).
- Há um acúmulo no teto: 6.160 chamados com 5,0 contra 4.363 com 4,9, quando a curva já descia. É compatível com notas acima de 5 cortadas em 5,0 (hipótese, sem ação).

In [0]:
n = (at_dt.withColumn("nota_txt", F.col("nota_satisfacao").cast("string"))
          .withColumn("nota_num", F.expr("try_cast(replace(nota_txt, ',', '.') as double)")))

print("nota preenchida:", n.filter("nota_txt IS NOT NULL").count(), "| nula:", n.filter("nota_txt IS NULL").count())
print("nota preenchida que não vira número:")
n.filter("nota_txt IS NOT NULL AND nota_num IS NULL").groupBy("nota_txt").count().orderBy(F.desc("count")).show(20)
print("nota com vírgula:", n.filter(F.col("nota_txt").contains(",")).count())

n.agg(F.min("nota_num"), F.max("nota_num"), F.round(F.avg("nota_num"), 3)).display()
(n.withColumn("nota_arred", F.round("nota_num", 1))
  .groupBy("nota_arred").count().orderBy("nota_arred").show(60))

nota preenchida: 309305 | nula: 9566
nota preenchida que não vira número:
+--------+-----+
|nota_txt|count|
+--------+-----+
+--------+-----+

nota com vírgula: 0


min(nota_num),max(nota_num),"round(avg(nota_num), 3)"
1.3,5.0,4.043


+----------+-----+
|nota_arred|count|
+----------+-----+
|      NULL| 9566|
|       1.3|    1|
|       1.4|    1|
|       1.5|    3|
|       1.6|    3|
|       1.7|    8|
|       1.8|   21|
|       1.9|   31|
|       2.0|   70|
|       2.1|  106|
|       2.2|  185|
|       2.3|  289|
|       2.4|  481|
|       2.5|  610|
|       2.6|  880|
|       2.7| 1174|
|       2.8| 1451|
|       2.9| 1910|
|       3.0| 2341|
|       3.1| 3040|
|       3.2| 4346|
|       3.3| 5782|
|       3.4| 7978|
|       3.5|10846|
|       3.6|14649|
|       3.7|18156|
|       3.8|22131|
|       3.9|25259|
|       4.0|27168|
|       4.1|27893|
|       4.2|27224|
|       4.3|25040|
|       4.4|21551|
|       4.5|17841|
|       4.6|13465|
|       4.7| 9991|
|       4.8| 6857|
|       4.9| 4363|
|       5.0| 6160|
+----------+-----+



In [0]:
# Nota x chamado aberto ou fechado
(n.groupBy("aberto")
  .agg(F.count("*").alias("chamados"),
       F.sum(F.col("nota_num").isNotNull().cast("int")).alias("com_nota"))
  .display())

aberto,chamados,com_nota
false,318871,309305


## 9. Histórico (tratativas)

**O que explora:** abre o `historico` em uma linha por tratativa e confere:
- `sequencia` começa em 1, não tem buraco nem repetição e acompanha a posição na lista;
- as datas das tratativas estão em ordem crescente;
- cada tratativa cai entre a abertura e o fechamento do chamado;
- a última ação de um chamado fechado é diferente da de um chamado aberto?

**Como ler:** o grão proposto para a Silver é uma tabela por chamado e outra por tratativa, sem listas. Os resultados abaixo dizem se a ordem do histórico é confiável ou se a Silver precisa reordenar pela data.

**Achado:** histórico consistente.
- 636.857 tratativas. `sequencia` sempre vira número e é igual à posição + 1. As datas convertem e estão em ordem.
- Nenhuma tratativa antes da abertura ou depois do fechamento.
- **A última ação não indica o status:** todos os chamados estão fechados, mas a última ação se divide entre as cinco (DESPACHO 64.257, RETORNO AO CLIENTE 63.987, ABERTURA 63.903, ENCERRAMENTO 63.432, ANALISE 63.292). O histórico é uma amostra de ações, e não um processo completo, então `ultima_acao` não deve ser lida como estado do chamado.

In [0]:
tr = (limpo.select("protocolo", F.posexplode_outer("historico").alias("pos", "t"))
           .filter("t IS NOT NULL")
           .select("protocolo", "pos", "t.*"))
tr.printSchema()

w = Window.partitionBy("protocolo").orderBy("pos")
tr = (tr.withColumn("ts", F.expr("try_cast(data as timestamp)"))
        .withColumn("seq_num", F.expr("try_cast(sequencia as int)"))
        .withColumn("ts_ant", F.lag("ts").over(w)))

print("tratativas:", f"{tr.count():,}")
print("sequencia que não vira número:", tr.filter("sequencia IS NOT NULL AND seq_num IS NULL").count())
print("sequencia diferente de posição + 1:", tr.filter(F.col("seq_num") != F.col("pos") + 1).count())
print("data da tratativa não convertida:", tr.filter("data IS NOT NULL AND ts IS NULL").count())
print("datas fora de ordem (tratativa anterior mais nova):", tr.filter("ts < ts_ant").count())

root
 |-- protocolo: string (nullable = true)
 |-- pos: integer (nullable = true)
 |-- acao: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- data: string (nullable = true)
 |-- sequencia: string (nullable = true)

tratativas: 636,857
sequencia que não vira número: 0
sequencia diferente de posição + 1: 0
data da tratativa não convertida: 0
datas fora de ordem (tratativa anterior mais nova): 0


In [0]:
# Tratativas dentro do período do chamado
td = tr.join(at_dt.select("protocolo", "abertura_ts", "fechamento_ts", "aberto"), "protocolo")
print("tratativa antes da abertura:", td.filter("ts < abertura_ts").count())
print("tratativa depois do fechamento:", td.filter("fechamento_ts IS NOT NULL AND ts > fechamento_ts").count())

# Última ação x chamado aberto ou fechado
ult = (td.withColumn("r", F.row_number().over(Window.partitionBy("protocolo").orderBy(F.desc("pos"))))
         .filter("r = 1"))
ult.groupBy("aberto", "acao").count().orderBy("aberto", F.desc("count")).display()

tratativa antes da abertura: 0
tratativa depois do fechamento: 0


aberto,acao,count
false,DESPACHO,64257
false,RETORNO AO CLIENTE,63987
false,ABERTURA,63903
false,ENCERRAMENTO,63432
false,ANALISE,63292


## 10. Distribuição no tempo e por UC

**O que explora:** quantos chamados abrem por mês e quantos chamados cada UC tem. Serve para a `kpis_mensal` (chamados por mês e nota média) e para a visão 360 (chamados já abertos da UC).

**Como ler:** em `faturas` e `leituras` os defeitos eram uniformes no tempo, e o volume por mês, regular. Se `atendimentos` tiver picos ou meses vazios, vale investigar antes de a Gold usar. Muitas UCs sem nenhum chamado é normal.

**Achado:**
- Cada UC tem em média 7,1 chamados (mediana 7, p90 11, p99 14, máximo 20), e 44.940 UCs têm ao menos um.
- O volume mensal é regular, ~6.400 chamados (de 6.237 a 6.624), **com uma exceção: 2025-07, com 18.061 chamados (2,8×)**. É o primeiro mês dos "últimos 12 meses" do diagnóstico.
- Entre o primeiro ano (76.561 chamados) e os últimos 12 meses (88.823) a alta é de **16%**. Sem o excesso de julho (~11.660 chamados), seria de ~0,8%.

In [0]:
(at_dt.withColumn("mes_abertura", F.date_format("abertura_ts", "yyyy-MM"))
      .groupBy("mes_abertura").count().orderBy("mes_abertura").display())

mes_abertura,count
2022-07,6322
2022-08,6474
2022-09,6309
2022-10,6326
2022-11,6467
2022-12,6377
2023-01,6323
2023-02,6448
2023-03,6315
2023-04,6482


In [0]:
por_uc = at_dt.groupBy("id_uc").count()
por_uc.agg(F.count("*").alias("ucs_com_chamado"),
           F.round(F.avg("count"), 2).alias("media"),
           F.expr("percentile_approx(count, array(0.5, 0.9, 0.99))").alias("mediana_p90_p99"),
           F.max("count").alias("maximo")).display()

ucs_com_chamado,media,mediana_p90_p99,maximo
44940,7.1,"List(7, 11, 14)",20


### 10.1 O pico de julho de 2025 nos chamados

**O que explora:** em que assuntos, canais, dias e UCs o excesso de julho se concentra. Compara cada categoria com a **média mensal dos outros 47 meses**; `razao` é julho dividido por essa média.

**Como ler:** se só um assunto ou canal multiplicasse, seria um evento específico. Se tudo sobe junto, o aumento é geral.

**Achado:** o aumento é **geral**.
- Todos os assuntos sobem entre 2,67× e 3,06× (`FALTA DE ENERGIA` 3,06×; `LIGACAO NOVA`, `RELIGACAO`, `FATURA`, `DANO ELETRICO` e `GERACAO DISTRIBUIDA`, de 2,67× a 2,75×).
- Todos os subassuntos sobem entre 2,61× e 3,11× (`OSCILACAO`, `INTERRUPCAO TOTAL` e `INTERRUPCAO PARCIAL`, ~3,0×; os demais ~2,7×).
- Todos os canais sobem entre 2,79× e 2,88×.
- Os 31 dias de julho têm volume parecido (de 535 a 629 chamados por dia), sem dia de pico.
- 18.061 chamados em **14.828 UCs distintas**: espalhado, e não concentrado em poucas UCs.

In [0]:
base_mes = at_dt.withColumn("mes_abertura", F.date_format("abertura_ts", "yyyy-MM"))
jul = base_mes.filter("mes_abertura = '2025-07'")
outros = base_mes.filter("mes_abertura != '2025-07'")

# 1. O que sobe em julho de 2025, comparado com a média mensal dos outros 47 meses?
for col in ["assunto", "subassunto", "canal"]:
    a = jul.groupBy(col).count().withColumnRenamed("count", "jul_2025")
    b = outros.groupBy(col).count().withColumn("media_outros", F.round(F.col("count") / 47, 0)).drop("count")
    print(col)
    a.join(b, col).withColumn("razao", F.round(F.col("jul_2025") / F.col("media_outros"), 2)).orderBy(F.desc("razao")).show(20, truncate=False)

# 2. Em que dias de julho?
jul.groupBy(F.to_date("abertura_ts").alias("dia")).count().orderBy("dia").show(31)

# 3. São as mesmas UCs ou espalhado? (chamados / UCs distintas)
print("chamados:", jul.count(), "| UCs distintas:", jul.select("id_uc").distinct().count())

assunto
+-------------------+--------+------------+-----+
|assunto            |jul_2025|media_outros|razao|
+-------------------+--------+------------+-----+
|FALTA DE ENERGIA   |6186    |2024.0      |3.06 |
|LIGACAO NOVA       |2406    |875.0       |2.75 |
|RELIGACAO          |2385    |871.0       |2.74 |
|FATURA             |2384    |877.0       |2.72 |
|DANO ELETRICO      |2361    |877.0       |2.69 |
|GERACAO DISTRIBUIDA|2339    |876.0       |2.67 |
+-------------------+--------+------------+-----+

subassunto
+-------------------+--------+------------+-----+
|subassunto         |jul_2025|media_outros|razao|
+-------------------+--------+------------+-----+
|OSCILACAO          |2088    |671.0       |3.11 |
|INTERRUPCAO TOTAL  |2042    |673.0       |3.03 |
|INTERRUPCAO PARCIAL|2056    |680.0       |3.02 |
|PRAZO              |805     |290.0       |2.78 |
|RESSARCIMENTO      |1210    |436.0       |2.78 |
|CREDITOS           |804     |289.0       |2.78 |
|APOS PAGAMENTO     |1198    |

### 10.2 O pico em outras tabelas: rede, consumo e comportamento dos chamados

**O que explora:** (1) se as interrupções de rede (`ocorrencias_rede`) também sobem em 2025-07; (2) se o protocolo acompanha a data (os chamados de julho têm uma faixa contígua ou foram inseridos depois); (3) se os chamados de julho se comportam como os dos outros meses (tempo até fechar, nota); (4) se o consumo de julho foge do padrão.

**Achado:** o pico aparece em **três tabelas** e aponta para um evento de rede em julho de 2025.

| Julho/2025 | Valor | Normal | Razão |
|---|---|---|---|
| Ocorrências de rede | 17.844 | ~3.370 por mês | **5,3×** |
| Chamados | 18.061 | ~6.400 por mês | **2,8×** |
| Tempo até fechar o chamado | 361,1 h | 119,9 h | **3,0×** |
| Nota média | 3,107 | 4,099 | **−1,0 ponto** |
| Chamados sem nota | 3,13% | 2,99% | igual |

- **Os protocolos são contíguos e acompanham o tempo:** cada mês ocupa uma faixa própria, sem buracos nem sobreposição. Julho vai de 230.048 a 248.108, exatamente 18.061 números seguidos. Os chamados extras foram gerados em sequência, e não acrescentados depois.
- As 180.000 ocorrências de rede batem com o case ("180 mil"), todas com data no formato ISO.
- **O consumo de julho é normal:** 13,49 milhões de kWh, 3,2% abaixo de julho de 2024, na mesma linha dos outros meses. O evento **não** explica a queda de volume. O consumo é fortemente sazonal (pico em janeiro e fevereiro, ~22 milhões de kWh; vale em julho e agosto, ~14 milhões), então só se compara o mesmo mês entre anos.

In [0]:
# 1. As interrupções da rede também sobem em 2025-07? (ocorrencias_rede, só linhas com data ISO)
oc = spark.table(f"{bronze_schema}.ocorrencias_rede")
eh_iso = F.col("data_abertura").rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2}")
(oc.filter(eh_iso).withColumn("mes", F.substring("data_abertura", 1, 7))
   .groupBy("mes").count().orderBy("mes").show(60))
print("ocorrências com data fora do formato ISO (não contadas):", oc.filter(~eh_iso).count())

# 2. O protocolo acompanha a data? Os chamados de julho têm uma faixa própria de protocolos?
(base_mes.withColumn("num", F.expr("try_cast(substring(protocolo, 3) as bigint)"))
   .groupBy("mes_abertura")
   .agg(F.min("num").alias("prot_min"), F.max("num").alias("prot_max"), F.count("*").alias("chamados"))
   .orderBy("mes_abertura").show(60))

# 3. Os chamados de julho se comportam como os dos outros meses?
(base_mes.withColumn("horas", (F.col("fechamento_ts").cast("long") - F.col("abertura_ts").cast("long")) / 3600)
   .withColumn("nota_num", F.expr("try_cast(nota_satisfacao as double)"))
   .withColumn("grupo", F.when(F.col("mes_abertura") == "2025-07", "jul_2025").otherwise("outros"))
   .groupBy("grupo")
   .agg(F.count("*").alias("chamados"),
        F.round(F.avg("horas"), 1).alias("horas_ate_fechar"),
        F.round(F.avg("nota_num"), 3).alias("nota_media"),
        F.round(F.avg(F.col("nota_num").isNull().cast("int")) * 100, 2).alias("pct_sem_nota"))
   .show())

+-------+-----+
|    mes|count|
+-------+-----+
|2022-07| 3333|
|2022-08| 3341|
|2022-09| 3415|
|2022-10| 3274|
|2022-11| 3433|
|2022-12| 3372|
|2023-01| 3336|
|2023-02| 3318|
|2023-03| 3293|
|2023-04| 3343|
|2023-05| 3312|
|2023-06| 3402|
|2023-07| 3398|
|2023-08| 3387|
|2023-09| 3424|
|2023-10| 3366|
|2023-11| 3330|
|2023-12| 3392|
|2024-01| 3450|
|2024-02| 3376|
|2024-03| 3399|
|2024-04| 3357|
|2024-05| 3443|
|2024-06| 3227|
|2024-07| 3354|
|2024-08| 3395|
|2024-09| 3374|
|2024-10| 3310|
|2024-11| 3286|
|2024-12| 3384|
|2025-01| 3413|
|2025-02| 3387|
|2025-03| 3400|
|2025-04| 3434|
|2025-05| 3411|
|2025-06| 3354|
|2025-07|17844|
|2025-08| 3409|
|2025-09| 3289|
|2025-10| 3407|
|2025-11| 5317|
|2025-12| 3403|
|2026-01| 3268|
|2026-02| 5302|
|2026-03| 3384|
|2026-04| 3326|
|2026-05| 3451|
|2026-06| 3377|
+-------+-----+

ocorrências com data fora do formato ISO (não contadas): 0
+------------+--------+--------+--------+
|mes_abertura|prot_min|prot_max|chamados|
+------------+--------+-

In [0]:
# 4. O consumo total de julho de 2025 foge do padrão? (consumo mensal; compare sempre o mesmo mês entre anos)
(spark.table(f"{silver_schema}.leituras").groupBy("mes_referencia")
      .agg(F.sum("consumo_kwh").alias("kwh")).orderBy("mes_referencia").show(60))

+--------------+--------+
|mes_referencia|     kwh|
+--------------+--------+
|       2022-07|14099929|
|       2022-08|14166862|
|       2022-09|15560451|
|       2022-10|18433287|
|       2022-11|20854327|
|       2022-12|21833062|
|       2023-01|22714349|
|       2023-02|22022368|
|       2023-03|22233809|
|       2023-04|20915567|
|       2023-05|18437672|
|       2023-06|15781244|
|       2023-07|14130835|
|       2023-08|14232008|
|       2023-09|15548157|
|       2023-10|18372295|
|       2023-11|20935107|
|       2023-12|21853392|
|       2024-01|22692225|
|       2024-02|21931552|
|       2024-03|22063337|
|       2024-04|20746382|
|       2024-05|18277653|
|       2024-06|15582546|
|       2024-07|13926072|
|       2024-08|13965720|
|       2024-09|15215237|
|       2024-10|17925510|
|       2024-11|20283219|
|       2024-12|21153741|
|       2025-01|22008862|
|       2025-02|21272995|
|       2025-03|21414448|
|       2025-04|20103053|
|       2025-05|17658944|
|       2025

### 10.3 Onde a rede sofreu em julho de 2025, e os picos menores

**O que explora:** causa e alimentadores das ocorrências de julho de 2025, e dos dois outros meses em que a rede teve um pico menor (2025-11 e 2026-02).

**Achado:**
- **Julho de 2025 é um evento climático e localizado.** 17.352 das 17.844 ocorrências (97,2%) têm causa `CLIMATICA`, contra ~560 por mês normalmente; o único tipo é `FALTA DE ENERGIA`. **Três alimentadores concentram 72%:** `ALM-REC-008` (6.887), `ALM-SAO-004` (3.351) e `ALM-JAB-004` (2.598). Os demais ficam em ~55 cada. Por dia, o volume é parecido (de 541 a 628), ou seja, não foi uma tempestade de um dia só.
- **2025-11 (5.317) e 2026-02 (5.302)** têm ~1,6× o normal, mas o aumento é **geral**: todas as causas sobem juntas (de 798 a 894 cada, nas seis causas), e os alimentadores mais afetados ficam em 56 a 68, sem concentração. Nesses meses os chamados **não** sobem (6.349 e 6.485). Os dois meses coincidem com consumo ~8% acima do ano anterior (ver `analise_reconciliacao_diretor`).
- **Defeito de dado em `ocorrencias_rede.causa`** (para o bloco A): a ausência aparece de quatro formas, o texto `NULL`, `-`, `N/A` e o valor nulo, cada uma com ~26 ocorrências por mês normal (~3% do total).

In [0]:
oc2 = oc.withColumn("mes", F.substring("data_abertura", 1, 7))
jul_oc, outros_oc = oc2.filter("mes = '2025-07'"), oc2.filter("mes != '2025-07'")

# Causa e tipo das ocorrências de julho/2025, contra a média mensal dos outros 47 meses
for col in ["causa", "tipo"]:
    a = jul_oc.groupBy(col).count().withColumnRenamed("count", "jul_2025")
    b = outros_oc.groupBy(col).count().withColumn("media_outros", F.round(F.col("count") / 47, 0)).drop("count")
    print(col)
    a.join(b, col).withColumn("razao", F.round(F.col("jul_2025") / F.col("media_outros"), 2)).orderBy(F.desc("razao")).show(20, truncate=False)

print("alimentadores com mais ocorrências em julho/2025")
jul_oc.groupBy("alimentador").count().orderBy(F.desc("count")).show(10)
jul_oc.groupBy(F.substring("data_abertura", 1, 10).alias("dia")).count().orderBy("dia").show(31)

causa
+---------+--------+------------+-----+
|causa    |jul_2025|media_outros|razao|
+---------+--------+------------+-----+
|CLIMATICA|17352   |560.0       |30.99|
|NULL     |129     |26.0        |4.96 |
|-        |119     |26.0        |4.58 |
|N/A      |117     |26.0        |4.5  |
+---------+--------+------------+-----+

tipo
+----------------+--------+------------+-----+
|tipo            |jul_2025|media_outros|razao|
+----------------+--------+------------+-----+
|FALTA DE ENERGIA|17844   |2207.0      |8.09 |
+----------------+--------+------------+-----+

alimentadores com mais ocorrências em julho/2025
+-----------+-----+
|alimentador|count|
+-----------+-----+
|ALM-REC-008| 6887|
|ALM-SAO-004| 3351|
|ALM-JAB-004| 2598|
|ALM-PAU-007|   59|
|ALM-JAB-020|   56|
|ALM-REC-027|   56|
|ALM-IPO-001|   55|
|ALM-JAB-016|   54|
|ALM-PAU-009|   54|
|ALM-REC-033|   53|
+-----------+-----+
only showing top 10 rows
+----------+-----+
|       dia|count|
+----------+-----+
|2025-07-01|  610|
|2

In [0]:
# Os dois picos menores: 2025-11 e 2026-02 (causa e alimentadores)
for m in ["2025-11", "2026-02"]:
    print(m)
    oc2.filter(F.col("mes") == m).groupBy("causa").count().orderBy(F.desc("count")).show(10, truncate=False)
    oc2.filter(F.col("mes") == m).groupBy("alimentador").count().orderBy(F.desc("count")).show(5)

2025-11
+--------------------+-----+
|causa               |count|
+--------------------+-----+
|CLIMATICA           |894  |
|NAO IDENTIFICADA    |879  |
|VEGETACAO           |877  |
|TERCEIROS           |849  |
|SOBRECARGA          |846  |
|FALHA DE EQUIPAMENTO|798  |
|-                   |48   |
|N/A                 |45   |
|NULL                |44   |
|NULL                |37   |
+--------------------+-----+

+-----------+-----+
|alimentador|count|
+-----------+-----+
|ALM-REC-017|   68|
|ALM-REC-032|   60|
|ALM-JAB-015|   59|
|ALM-CAM-004|   59|
|ALM-ILH-001|   57|
+-----------+-----+
only showing top 5 rows
2026-02
+--------------------+-----+
|causa               |count|
+--------------------+-----+
|FALHA DE EQUIPAMENTO|880  |
|SOBRECARGA          |864  |
|VEGETACAO           |853  |
|TERCEIROS           |848  |
|NAO IDENTIFICADA    |841  |
|CLIMATICA           |836  |
|N/A                 |48   |
|NULL                |46   |
|NULL                |44   |
|-                   |42 

## 11. Resumo e regras propostas para a Silver

`atendimentos` é a tabela mais limpa do projeto: não há cópias, órfãs, cabeçalho repetido, datas misturadas nem sentinelas. A Silver precisa de poucos tratamentos, e a maior parte vira checagem de proteção com contagem esperada zero. A coluna "Decisão" é da responsável pela tabela e deve ser registrada no `data_quality_log.md` com alternativa considerada, impacto na Gold e quem decidiu.

| Problema | Linhas | Regra proposta | Decisão |
|---|---|---|---|
| Cópias exatas por protocolo | 0 | manter `dropDuplicates(["protocolo"])` como proteção, com contagem esperada 0 | proposta |
| Cabeçalho repetido | 0 | manter o filtro como proteção (a tabela vem de JSON, mas `faturas` e `unidades_consumidoras` tiveram) | proposta |
| UC nula, inexistente ou de teste | 0 | manter a validação contra `silver.unidades_consumidoras` como proteção | proposta |
| Datas em mais de um formato | 0 | converter com `try_to_timestamp`, formato único `yyyy-MM-dd HH:mm:ss` (e `T` no histórico) | proposta |
| `data_fechamento` nula (chamado aberto) | 0 hoje | manter a flag `aberto` para cargas futuras (constante falsa hoje) | proposta |
| Nota nula | 9.566 (3,0%) | **manter nula** (chamado não avaliado, nulo de negócio) | proposta |
| Nota em formato texto ou fora de 1 a 5 | 0 | `try_cast` para decimal; checagem de faixa de 1 a 5 | proposta |
| Nota acumulada no teto de 5,0 | 6.160 em 5,0 | não alterar; documentar como hipótese de corte no dicionário | proposta |
| Categorias com grafias diferentes | 0 | `upper(trim)` por segurança | proposta |
| Histórico: ordem, sequência e datas | 0 problemas | abrir em `silver.tratativas_atendimento`, com chave `(protocolo, sequencia)` | proposta |
| `ultima_acao` não indica o status | — | não derivar status de `ultima_acao`; documentar no dicionário | proposta |
| **Pico de chamados em 2025-07 (2,8×)** | ~11.660 a mais | **manter tudo** (dado consistente, protocolos contíguos); documentar como evento de rede de julho de 2025 | proposta |

**Para a Silver:** seguir o molde de `02_silver_leituras` e `02_silver_faturas` (markdown antes de cada tratamento, `dq_check` com contagem esperada, gravação só se todas as checagens passarem, `dq_log` gravado sempre e comparação Bronze × Silver com os mesmos ids). Contagens esperadas: 318.871 chamados e 636.857 tratativas. Grãos: `silver.atendimentos` (um chamado por linha, chave `protocolo`) e `silver.tratativas_atendimento` (uma tratativa por linha, chave `(protocolo, sequencia)`).

**Para a Gold e o dicionário de dados:** `qtd_atendimentos` e `nota_satisfacao_media` por mês mostram o pico de 2025-07, e a variação de chamados entre os dois períodos (+16%) vem quase toda desse mês. O dicionário deve registrar isso.

**Para o bloco A:** `ocorrencias_rede.causa` tem ausência escrita de quatro formas (`NULL`, `-`, `N/A` e nulo), e há um evento de rede em 2025-07 (17.844 ocorrências, 97% `CLIMATICA`, 72% em três alimentadores).

**Esta exploração está concluída.** Novos achados devem ir para `docs/data_quality_log.md`.